# Week 1 · Mia's moving day as a model

**Today's sentence:** productivity is always a ratio, and the interesting question is what goes into the denominator.

This morning you calculated Plans A and B by hand. Now you build the same calculation as a small computer model. A model is a calculator you write once and then ask many questions: *what if we take the large van? a fourth friend? pack half the boxes early?* Answering these by hand would take an hour each; the model answers in a second.

**How this notebook works**

- Google Colab is a document in your browser in which text and small pieces of Python code sit side by side. You run a code cell with the ▶ button on its left, or with **Shift + Enter**.
- Run the cells **from top to bottom**. A cell can only use what the cells above it have created.
- Cells marked **✏️ YOUR TURN** contain `...` or a `# TODO`. Replace them with your code.
- Every step ends with a **check cell**. It prints ✅ when your result matches this morning's hand calculation, and ❌ with a hint when it does not.
- You may use an AI assistant. At the end you write one sentence on what you checked in its answer.

All rules and numbers are those of Handout 2 (the data sheet).

## Step 0 · Setup

This cell loads three libraries. A library is a toolbox other people have written, so we do not have to: `pandas` for tables, `numpy` for numbers, `matplotlib` for charts. Run it once.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def check(label, got, expected, tol=0.01, hint=""):
    """Compare your result with the hand calculation and print ✅ or ❌."""
    try:
        ok = abs(float(got) - float(expected)) <= tol
    except Exception:
        ok = got == expected
    print(("✅ " if ok else "❌ ") + f"{label}: you have {got}, expected {expected}" + ("" if ok else f"   Hint: {hint}"))

def clock(minutes_after_nine):
    """Turn 'minutes after 09:00' into a clock time such as '17:25'."""
    m = int(round(minutes_after_nine))
    return f"{9 + m // 60:02d}:{m % 60:02d}"

print("Setup done. Example:", clock(505), "is 505 minutes after 09:00.")

## Step 1 · The task table

A model starts with its data. We store Handout 2's task table as a **DataFrame**: a table inside Python, with named columns, like a small spreadsheet.

**✏️ YOUR TURN:** three rows are filled in. Fill in the missing numbers from Handout 2 (work in person-minutes and the maximum number of people).

In [ ]:
tasks = pd.DataFrame({
    "task":       ["pack", "dismantle", "carry", "load", "unload", "reassemble", "clean"],
    "work":       [360,     90,          ...,     ...,    ...,      ...,          120],   # person-minutes  # TODO
    "max_people": [3,       2,           ...,     ...,    ...,      ...,          2],     # TODO
}).set_index("task")

DRIVE = 15          # minutes for one drive between the flats, one driver
tasks

In [ ]:
# CHECK step 1
try:
    check("total work in person-minutes", int(tasks["work"].sum()), 1185, hint="add up the work column of Handout 2")
    check("carry: max people", int(tasks.loc["carry", "max_people"]), 4, hint="carrying down is the only task with four people")
except Exception as e:
    print("❌ The table is not complete yet:", e)

## Step 2 · How long does one task take?

Rule 3 and 4 of Handout 2: time = work ÷ number of people working, and never more people than the task's maximum. In Python, `min(a, b)` gives the smaller of two numbers, which is exactly the cap we need.

**✏️ YOUR TURN:** complete the function. A function is a small recipe with a name: you give it ingredients (here: the task, the crew size and the share of the work) and it gives back one number.

In [ ]:
def task_minutes(task, crew, share=1.0):
    """Minutes a task takes with a given crew. share = the part of the task done now (e.g. 0.5 per trip)."""
    work = tasks.loc[task, "work"] * share
    people = ...          # TODO: the crew, but never more than tasks.loc[task, "max_people"]
    return work / people

In [ ]:
# CHECK step 2
try:
    check("carry with 3 people", task_minutes("carry", 3), 90, hint="270 ÷ 3")
    check("carry with 5 people", task_minutes("carry", 5), 67.5, hint="only 4 people can carry: 270 ÷ 4")
    check("pack half the boxes with 3 people", task_minutes("pack", 3, share=0.5), 60, hint="180 ÷ 3")
except Exception as e:
    print("❌ Not working yet:", e)

## Step 3 · What does the day cost?

Handout 2: van day price + kilometres × price per km + 15 € per friend (Mia is free). A small van drives 2 trips (60 km), a large van 1 trip (30 km). We store the van data in a **dictionary**: a lookup list where a name (`"small"`) points to its values.

**✏️ YOUR TURN:** complete the three lines marked TODO.

In [ ]:
VANS = {
    "small": {"day": 45, "per_km": 0.30, "trips": 2},
    "large": {"day": 75, "per_km": 0.40, "trips": 1},
}
FRIEND = 15        # € for pizza and drinks per friend
BUDGET = 120

def day_cost(van, crew, late_slot=False):
    """Cost in € of van, friends and (optionally) the return slot until 20:00."""
    v = VANS[van]
    km = ...                      # TODO: 30 km per trip
    friends = ...                 # TODO: everyone in the crew except Mia
    cost = v["day"] + v["per_km"] * km + FRIEND * friends
    if late_slot:
        cost += 20
    return cost

In [ ]:
# CHECK step 3
try:
    check("Plan A cost (small van, crew 3)", day_cost("small", 3), 93, hint="45 + 0.30 × 60 + 2 × 15")
    check("Plan B cost (large van, crew 3)", day_cost("large", 3), 117, hint="75 + 0.40 × 30 + 2 × 15")
except Exception as e:
    print("❌ Not working yet:", e)

## Step 4 · The schedule: when is the day over?

Now the rules of the day, task after task, exactly as you did it this morning. The function walks through the tasks, adds up the minutes and writes every task into a list with its start and end. That list is also what we draw in Step 6.

Two things are special: the small van needs **two trips** (so carrying, loading and unloading are split in halves, and the van drives back in between), and **at the end** the crew splits: with 3 or more people one person drives the van back and cleans alone while the others reassemble; the day ends when the later of the two is done.

**✏️ YOUR TURN:** most of the function is written. Complete the two lines marked TODO.

In [ ]:
def schedule(van, crew, packed_early=0.0, work=None, drive=DRIVE):
    """Return (finish, plan): finish in minutes after 09:00, plan = list of (task, who, start, end).
    packed_early = share of the boxes Mia packed in the week before (0, 0.5)."""
    w = work if work is not None else tasks["work"].to_dict()     # the work numbers to use
    cap = tasks["max_people"].to_dict()
    def minutes(task, n, share=1.0):
        return w[task] * share / min(n, cap[task])
    plan, t = [], 0.0
    def add(task, who, duration, start):
        plan.append((task, who, start, start + duration))
        return start + duration

    t = add("pack", "crew", minutes("pack", crew, 1 - packed_early), t)
    t = add("dismantle", "crew", minutes("dismantle", crew), t)
    trips = VANS[van]["trips"]
    for k in range(trips):
        s = 1 / trips                                            # each trip moves this share of the flat
        t = add(f"carry {k+1}", "crew", minutes("carry", crew, s), t)
        t = add(f"load {k+1}", "crew", ..., t)                   # TODO: same idea as carry
        t = add(f"drive {k+1}", "driver", drive, t)
        t = add(f"unload {k+1}", "crew", minutes("unload", crew, s), t)
        if k < trips - 1:
            t = add(f"drive back {k+1}", "driver", ..., t)        # TODO: how long is one drive?
    unload_end = t
    if crew >= 3:      # one person drives back and cleans alone, the others reassemble
        e = add("drive to depot", "driver", drive, unload_end)
        clean_end = add("clean", "driver", w["clean"] / 1, e)
        reas_end = add("reassemble", "others", minutes("reassemble", crew - 1), unload_end)
        finish = max(clean_end, reas_end)
    else:              # two people: reassemble together, then drive back and clean together
        e = add("reassemble", "crew", minutes("reassemble", crew), unload_end)
        e = add("drive to depot", "crew", drive, e)
        finish = add("clean", "crew", minutes("clean", crew), e)
    return finish, plan


In [ ]:
# CHECK step 4
try:
    fa, _ = schedule("small", 3, packed_early=0.0)
    fb, _ = schedule("large", 3, packed_early=0.5)
    check("Plan A finish", clock(fa), "17:25", hint="did you add the drive back between the two trips?")
    check("Plan B finish", clock(fb), "15:55", hint="with half packed early, packing takes 60 minutes")
except Exception as e:
    print("❌ Not working yet:", e)

## Step 5 · Productivity: three ratios, three answers

Productivity is output divided by input. The output is the same in every plan: **100 moving units** (boxes and furniture). The input changes with the question you ask:

- per **person-hour on Saturday**: crew × hours from 09:00 to the finish,
- per **person-hour for the whole project**: the same plus Mia's packing hours in the week before (6 hours × the share packed early),
- per **euro**: the cost of the day.

**✏️ YOUR TURN:** write the three ratios. Then look at the table: which plan wins depends on the denominator.

In [ ]:
UNITS = 100

def productivity(van, crew, packed_early=0.0):
    finish, _ = schedule(van, crew, packed_early)
    person_hours_day = crew * finish / 60
    packing_hours = 6 * packed_early
    cost = day_cost(van, crew)
    return {
        "finish": clock(finish),
        "cost €": cost,
        "person-hours Saturday": round(person_hours_day, 2),
        "units per person-hour, Saturday": ...,            # TODO
        "units per person-hour, whole project": ...,       # TODO
        "units per €": ...,                                # TODO
    }

pd.DataFrame({"Plan A": productivity("small", 3, 0.0), "Plan B": productivity("large", 3, 0.5)})

In [ ]:
# CHECK step 5
try:
    a, b = productivity("small", 3, 0.0), productivity("large", 3, 0.5)
    check("Plan A units per person-hour, Saturday", a["units per person-hour, Saturday"], 3.96, hint="100 ÷ 25.25")
    check("Plan B units per person-hour, whole project", b["units per person-hour, whole project"], 4.21, hint="100 ÷ (20.75 + 3)")
    check("Plan A units per €", a["units per €"], 1.08, hint="100 ÷ 93")
except Exception as e:
    print("❌ Not working yet:", e)

## Step 6 · Chart 1: the day as a Gantt chart

A Gantt chart draws every task as a bar on a time axis, so you see at a glance what happens when, what runs in parallel and what the day is waiting for. This cell is complete: run it, then change the plan in the first line and run it again.

In [ ]:
van, crew, packed_early = "small", 3, 0.0          # ← change me: "large", 4, 0.5 ...

finish, plan = schedule(van, crew, packed_early)
fig, ax = plt.subplots(figsize=(10, 4.5))
colours = {"crew": "tab:blue", "driver": "tab:orange", "others": "tab:green"}
for i, (task, who, start, end) in enumerate(plan):
    ax.barh(i, end - start, left=start, color=colours[who])
    ax.text(end + 3, i, f"{task} ({who})", va="center", fontsize=8)
ax.axvline(540, color="red", linestyle="--")
ax.text(542, len(plan) - 0.5, "18:00 van back", color="red", fontsize=8)
ax.set_yticks([])
ax.invert_yaxis()
ticks = range(0, 661, 60)
ax.set_xticks(list(ticks), [clock(m) for m in ticks])
ax.set_title(f"{van} van, crew of {crew}, {packed_early:.0%} packed early → done at {clock(finish)}, {day_cost(van, crew):.0f} €")
plt.tight_layout()
plt.show()

**Look at the chart:** which task is the last one to finish? Would a fourth person help that task? (Rule 6 and the max-people column give the answer.)

## Step 7 · Chart 2: every option at once

The model can now calculate all combinations: 2 vans × 4 crew sizes × 3 packing choices = 24 plans. A **loop** repeats the same calculation for every combination and collects the results in a table. The chart then puts cost against finish time: the good plans are bottom left (cheap and early), and the lines show the limits (120 € and 18:00).

In [ ]:
rows = []
for van in ["small", "large"]:
    for crew in [2, 3, 4, 5]:
        for packed_early in [0.0, 0.5, 1.0]:
            finish, _ = schedule(van, crew, packed_early)
            cost = day_cost(van, crew)
            feasible = cost <= BUDGET and 6 * packed_early <= 4      # Mia has only 4 free hours
            rows.append({"van": van, "crew": crew, "packed early": packed_early,
                         "cost €": cost, "finish": clock(finish), "finish_min": finish,
                         "minutes to spare": round(540 - finish), "feasible": feasible})
options = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(8, 5))
for feasible, marker in [(True, "o"), (False, "x")]:
    d = options[options["feasible"] == feasible]
    ax.scatter(d["finish_min"], d["cost €"], marker=marker, s=60,
               label="feasible" if feasible else "over budget or not enough packing hours")
for _, r in options[options["feasible"]].iterrows():
    ax.annotate(f"{r['van'][0].upper()}{r['crew']}·{r['packed early']:.0%}", (r["finish_min"], r["cost €"]),
                fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.axhline(BUDGET, color="grey", linestyle="--"); ax.text(ax.get_xlim()[0], BUDGET + 2, "budget 120 €", fontsize=8)
ax.axvline(540, color="red", linestyle="--"); ax.text(542, ax.get_ylim()[0] + 2, "18:00", color="red", fontsize=8)
ticks = range(360, 721, 60)
ax.set_xticks(list(ticks), [clock(m) for m in ticks])
ax.set_xlabel("finish time"); ax.set_ylabel("cost of the day (€)")
ax.set_title("Every plan: cost against finish time  (S = small van, L = large van, crew, share packed early)")
ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

options[options["feasible"]].sort_values("finish_min").drop(columns="finish_min")

### Questions for your team (write the answers here)

1. Which single change buys the most minutes per euro?
2. Where does adding people stop helping? Find the pair of plans in the table that shows it.
3. Which plan would you give Mia, and what does she give up with it?

*Your answers:*

1.
2.
3. 

## Optional extensions for quick teams

Pick one. They are not needed for Test 1.

### E1 · Bad luck: a plan is a distribution, not a number

Every task can go faster or slower than planned. Instead of one work number we take three (optimistic, realistic, pessimistic) and let the computer draw 500 random Saturdays. **Random numbers** in a model are how we ask "how often does this go wrong?". The `seed` makes the draw repeatable, so everyone in the room gets the same answer.

In [ ]:
three_point = {   # optimistic, realistic, pessimistic (person-minutes)
    "pack": (300, 360, 540), "dismantle": (75, 90, 130), "carry": (220, 270, 460), "load": (100, 120, 180),
    "unload": (75, 90, 125), "reassemble": (110, 135, 215), "clean": (105, 120, 180)}
DRIVE_3 = (15, 15, 40)

def random_day(van, crew, packed_early, rng):
    w = {k: rng.triangular(o, r, p) for k, (o, r, p) in three_point.items()}
    d = rng.triangular(*DRIVE_3) if DRIVE_3[0] < DRIVE_3[2] else DRIVE_3[1]
    return schedule(van, crew, packed_early, work=w, drive=d)[0]

van, crew, packed_early = "small", 3, 0.5        # ← try other plans
rng = np.random.default_rng(seed=1)
days = np.array([random_day(van, crew, packed_early, rng) for _ in range(500)])
plan_finish = schedule(van, crew, packed_early)[0]

plt.figure(figsize=(8, 4))
plt.hist(days, bins=30)
plt.axvline(540, color="red", linestyle="--", label="18:00")
plt.axvline(plan_finish, color="black", label=f"the plan: {clock(plan_finish)}")
plt.axvline(days.mean(), color="green", label=f"average day: {clock(days.mean())}")
ticks = range(360, 781, 60); plt.xticks(list(ticks), [clock(m) for m in ticks])
plt.legend(); plt.title(f"500 random Saturdays: late on {np.mean(days > 540):.0%} of the days"); plt.show()

Why is the average day later than the plan? Look at the three numbers per task: how far can a task go faster, and how far slower?

### E2 · The event cards

Use the Handout 4 cards as changes to the work numbers, e.g. *blocked stairwell*: carrying takes 70 % longer. Copy the work dictionary, change one number and run `schedule` again.

In [ ]:
w = tasks["work"].to_dict()
w["carry"] = w["carry"] * 1.7                      # card 1: blocked stairwell
finish, _ = schedule("small", 3, 0.0, work=w)
print("Plan A with a blocked stairwell:", clock(finish), "→ late" if finish > 540 else "→ still in time")

### E3 · Build your own: the party

Twenty guests arrive at 20:00. The day before: shop 60 min, yogurt tiramisu 25 min of work and then 6 hours in the fridge (no helper can shorten that), clean the flat 90 min. On the day: toppings 30 min, shape 40 patties 20 min, drinks 20 min in ice water, sound and decoration 40 min, smash burgers on **one** pan: 4 patties per batch, 5 min per batch.

Make a task table like Step 1, a function for the latest start time, and one chart. Does a third friend bring the start forward? What if the friend brings a second pan?

In [ ]:
party = pd.DataFrame({
    "task": ["shop", "tiramisu", "clean", "toppings", "patties", "drinks", "sound and decoration", "burgers"],
    "work": [60, 25, 90, 30, 20, 20, 40, 50],
    "max_people": [2, 1, 2, 2, 2, 1, 2, 1],
    "day": ["before", "before", "before", "party", "party", "party", "party", "party"],
})
party
# your model here

### E4 · Sliders

In Colab you can turn any input into a slider, so you can play with the plan without typing. Run the cell and move the sliders.

In [ ]:
from ipywidgets import interact

@interact(van=["small", "large"], crew=(2, 5), packed_early=[0.0, 0.5])
def play(van="small", crew=3, packed_early=0.0):
    p = productivity(van, crew, packed_early)
    for k, v in p.items():
        print(f"{k:40s} {v}")

## Before you leave · AI check

If you used an AI assistant today, write one sentence: **what did you check in its answer, and what did you change?**

*Your sentence:* 

---
*Source of the case, rules and numbers:* Eigene Darstellung, R. Bartnik (2026), Session 1 moving-day exercise, Handouts 1–4. Definitions of productivity: Heizer, J., Render, B., & Munson, C. (2023). *Operations management: Sustainability and supply chain management* (14th ed., Global ed.). Pearson, Chapter 1.

*Transparency note:* Claude was used to develop the notebook and as a reviewer for language and consistency. All numbers were checked against the hand calculations of the exercise key. All content was reviewed, revised and is the responsibility of Roman Bartnik.